# 64 — Classification Ensemble Experiments

**Why**: nb63 showed that stacking *regressors* into a LogReg meta-learner
closes only 0.22pp of the 7.4pp gap vs the classification baseline.
Regressors optimise the wrong objective (MSE on log-citations).

**This notebook**: Stack *classifiers* instead — each base model already
optimises the binary boundary, so their probability outputs are
directly meaningful for the meta-learner.

## Experiments

| # | Method | Scenario |
|---|--------|----------|
| 1 | Soft vote (avg probabilities) | A / B |
| 2 | Weighted soft vote (weight by CV F1) | A / B |
| 3 | Stacking: all classifiers -> LogReg | A / B |
| 4 | Stacking: top-2 (LogReg + XGB) -> LogReg | A / B |
| 5 | Cross-scenario: B classifiers -> A meta-LogReg | - |
| 6 | Domain-aware: ensemble within each domain | A |

**Baselines**:
- Direct LogReg (threshold=0.54): **F1=62.55%, AUC=81.04%**
- Best overall (selective domain hybrid, nb42): **F1=63.33%**

In [ ]:
import sys, re, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    roc_auc_score, f1_score, precision_score, recall_score
)
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')

PROJECT_ROOT = Path('../../').resolve()
sys.path.insert(0, str(PROJECT_ROOT))

DATA_PATH   = PROJECT_ROOT / 'data' / 'processed' / 'all_unis_cleaned.pkl'
REPORTS_DIR = PROJECT_ROOT / 'reports'
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
TRAIN_YEARS  = [2015, 2016, 2017]
TEST_YEARS   = [2018, 2019, 2020]
TFIDF_MAX    = 5000

BASELINE_F1  = 0.6255
BASELINE_AUC = 0.8104
BEST_F1      = 0.6333  # selective domain hybrid (nb42)

print('Imports OK')

## 1. Load Data

In [ ]:
df = pd.read_pickle(DATA_PATH)
inst_col = 'institution' if 'institution' in df.columns else 'Institution'
aub_mask = df[inst_col].str.upper().str.contains('AUB|BEIRUT', na=False)

df_aub       = df[aub_mask].copy()
df_train_aub = df_aub[df_aub['Year'].isin(TRAIN_YEARS)].dropna(subset=['Abstract','Citations']).copy()
df_test_aub  = df_aub[df_aub['Year'].isin(TEST_YEARS)].dropna(subset=['Abstract','Citations']).copy()

df_train_all = df[df['Year'].isin(TRAIN_YEARS)].dropna(subset=['Abstract','Citations']).copy()
df_test_all  = df[df['Year'].isin(TEST_YEARS)].dropna(subset=['Abstract','Citations']).copy()
df_test_aub_from_all = df_test_all[
    df_test_all[inst_col].str.upper().str.contains('AUB|BEIRUT', na=False)
].copy()

thr_A  = df_train_aub['Citations'].quantile(0.75)
thr_BC = df_train_all['Citations'].quantile(0.75)

print(f'Scenario A  -- Train (AUB): {len(df_train_aub):,} | Test (AUB): {len(df_test_aub):,}')
print(f'Scenario B  -- Train (merged): {len(df_train_all):,} | Test (AUB): {len(df_test_aub_from_all):,}')
print(f'p75 threshold  A: {thr_A:.0f} | B: {thr_BC:.0f}')

## 2. Feature Engineering

In [ ]:
def preprocess_text(text):
    if pd.isna(text): return ''
    return str(text).lower()

def build_tfidf_features(df_tr, dfs_te, max_features=TFIDF_MAX):
    tfidf = TfidfVectorizer(
        max_features=max_features, ngram_range=(1, 2),
        min_df=5, max_df=0.80, stop_words='english', sublinear_tf=True
    )
    abs_tr = df_tr['Abstract'].apply(preprocess_text)
    tfidf.fit(abs_tr)
    cols = [f'tfidf_{w}' for w in tfidf.get_feature_names_out()]
    tr_feat = pd.DataFrame.sparse.from_spmatrix(tfidf.transform(abs_tr), index=df_tr.index, columns=cols)
    te_feats = [
        pd.DataFrame.sparse.from_spmatrix(
            tfidf.transform(df_te['Abstract'].apply(preprocess_text)), index=df_te.index, columns=cols
        ) for df_te in dfs_te
    ]
    return tr_feat, te_feats

def build_venue_features(df_):
    vf = pd.DataFrame(index=df_.index)
    for feat, col in [
        ('snip',                 'SNIP (publication year)'),
        ('snip_percentile',      'SNIP percentile (publication year) *'),
        ('citescore',            'CiteScore (publication year)'),
        ('citescore_percentile', 'CiteScore percentile (publication year) *'),
        ('sjr',                  'SJR (publication year)'),
        ('sjr_percentile',       'SJR percentile (publication year) *'),
    ]:
        vf[feat] = pd.to_numeric(df_.get(col, pd.Series(np.nan, index=df_.index)), errors='coerce')
    vf['topic_prominence']         = pd.to_numeric(df_.get('Topic Prominence Percentile', pd.Series(np.nan, index=df_.index)), errors='coerce')
    vf['topic_cluster_prominence'] = pd.to_numeric(df_.get('Topic Cluster Prominence Percentile', pd.Series(np.nan, index=df_.index)), errors='coerce')
    vf['avg_venue_percentile']     = vf[['snip_percentile','citescore_percentile','sjr_percentile']].mean(axis=1)
    vf['is_top_journal']           = (vf['avg_venue_percentile'] >= 75).astype(float)
    return vf.fillna(vf.median(numeric_only=True))

def build_author_features(df_):
    af = pd.DataFrame(index=df_.index)
    for feat, col in [('num_authors','Number of Authors'),
                      ('num_institutions','Number of Institutions'),
                      ('num_countries','Number of Countries/Regions')]:
        af[feat] = pd.to_numeric(df_.get(col, pd.Series(np.nan, index=df_.index)), errors='coerce')
    af['is_international_collab'] = (af['num_countries'] > 1).astype(float)
    af['is_multi_inst']           = (af['num_institutions'] > 1).astype(float)
    return af.fillna(af.median(numeric_only=True))

def build_metadata_features(df_):
    mf = pd.DataFrame(index=df_.index)
    oa_col = 'Open Access' if 'Open Access' in df_.columns else 'is_open_access'
    mf['is_open_access'] = df_.get(oa_col, pd.Series(0, index=df_.index)).notna().astype(float)
    pub_type = df_.get('Publication Type', pd.Series('', index=df_.index)).fillna('').str.lower()
    mf['is_article']    = pub_type.str.contains('article').astype(float)
    mf['is_review']     = pub_type.str.contains('review').astype(float)
    mf['is_conference'] = pub_type.str.contains('conference|proceedings').astype(float)
    mf['pub_year']      = pd.to_numeric(df_.get('Year', pd.Series(2015, index=df_.index)), errors='coerce').fillna(2015)
    return mf

def build_numeric_features(df_):
    vf = build_venue_features(df_)
    af = build_author_features(df_)
    mf = build_metadata_features(df_)
    ix = pd.DataFrame({
        'top_journal_x_intl_collab':    vf['is_top_journal']       * af['is_international_collab'],
        'venue_pct_x_num_authors':      vf['avg_venue_percentile'] * af['num_authors'],
        'venue_pct_x_num_institutions': vf['avg_venue_percentile'] * af['num_institutions'],
        'snip_x_num_authors':           vf['snip']                 * af['num_authors'],
    }, index=df_.index)
    return pd.concat([vf, af, mf, ix], axis=1)

def drop_year_tokens(X):
    return X.drop(columns=[c for c in X.columns if re.match(r'tfidf_\d{4}$', str(c))], errors='ignore')

def assemble_features(df_tr, dfs_te):
    text_tr, text_tes = build_tfidf_features(df_tr, dfs_te)
    num_tr  = build_numeric_features(df_tr)
    num_tes = [build_numeric_features(d) for d in dfs_te]
    X_tr  = drop_year_tokens(pd.concat([text_tr,  num_tr],  axis=1))
    X_tes = [drop_year_tokens(pd.concat([tt, nn], axis=1)) for tt, nn in zip(text_tes, num_tes)]
    for i, X_te in enumerate(X_tes):
        for c in set(X_tr.columns) - set(X_te.columns):
            X_te[c] = 0
        X_tes[i] = X_te[X_tr.columns]
    return X_tr.fillna(0).astype('float32'), [x.fillna(0).astype('float32') for x in X_tes]

print('Feature functions defined.')

In [ ]:
print('Building Scenario A features...')
X_train_A, [X_test_A] = assemble_features(df_train_aub, [df_test_aub])
y_train_A = (df_train_aub['Citations'].values >= thr_A).astype(int)
y_test_A  = (df_test_aub['Citations'].values  >= thr_A).astype(int)
print(f'  X_train_A: {X_train_A.shape} | class balance: {y_train_A.mean():.1%} high')

print('Building Scenario B features...')
X_train_BC, [X_test_B] = assemble_features(df_train_all, [df_test_aub_from_all])
y_train_BC = (df_train_all['Citations'].values         >= thr_BC).astype(int)
y_test_B   = (df_test_aub_from_all['Citations'].values >= thr_BC).astype(int)
print(f'  X_train_BC: {X_train_BC.shape} | class balance: {y_train_BC.mean():.1%} high')

## 3. Base Classifiers

Four classifiers representing different inductive biases:
- **LogReg** — linear, strong baseline
- **XGBoost** — gradient boosted trees, captures non-linear interactions
- **LightGBM** — faster GBDT, different leaf-splitting strategy
- **RandomForest** — bagging, high variance reducer

All use class_weight='balanced' or equivalent to handle the 75/25 imbalance.

In [ ]:
pos_ratio = (1 - y_train_A.mean()) / y_train_A.mean()  # ~3 for 75/25 split

BASE_CLFS = {
    'LogReg': Pipeline([
        ('scaler', StandardScaler(with_mean=False)),
        ('clf',    LogisticRegression(max_iter=1000, random_state=RANDOM_STATE,
                                      class_weight='balanced'))
    ]),
    'XGBoost': XGBClassifier(
        n_estimators=500, learning_rate=0.05, max_depth=6,
        subsample=0.8, colsample_bytree=0.8,
        scale_pos_weight=pos_ratio,
        random_state=RANDOM_STATE, n_jobs=-1, verbosity=0, eval_metric='logloss'
    ),
    'LightGBM': LGBMClassifier(
        n_estimators=1000, learning_rate=0.03, num_leaves=63,
        min_child_samples=20, subsample=0.8, colsample_bytree=0.8,
        class_weight='balanced',
        random_state=RANDOM_STATE, n_jobs=-1, verbose=-1
    ),
    'RandomForest': RandomForestClassifier(
        n_estimators=300, max_depth=12, min_samples_leaf=10,
        class_weight='balanced',
        random_state=RANDOM_STATE, n_jobs=-1
    ),
}

print('Base classifiers:')
for name in BASE_CLFS:
    print(f'  {name}')

## 4. Train Base Classifiers & Collect OOF Probabilities

In [ ]:
import copy

def train_base_classifiers(X_tr, y_tr, X_te, scenario_label):
    oof_probs  = {}
    test_probs = {}
    cv_f1s     = {}
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

    for name, clf in BASE_CLFS.items():
        print(f'  [{scenario_label}] {name}...', end=' ')
        clf_ = copy.deepcopy(clf)
        # Out-of-fold probabilities (5-fold stratified)
        oof = cross_val_predict(clf_, X_tr, y_tr, cv=skf,
                                method='predict_proba', n_jobs=1)[:, 1]
        oof_probs[name] = oof
        cv_f1 = f1_score(y_tr, (oof >= 0.5).astype(int), zero_division=0)
        cv_f1s[name] = cv_f1
        # Full-fit for test predictions
        clf_.fit(X_tr, y_tr)
        test_probs[name] = clf_.predict_proba(X_te)[:, 1]
        # Individual test F1 (default threshold)
        test_f1 = f1_score(y_te_map[scenario_label],
                           (test_probs[name] >= 0.5).astype(int), zero_division=0)
        print(f'OOF F1={cv_f1:.4f} | Test F1={test_f1:.4f}')

    return oof_probs, test_probs, cv_f1s

# Map scenario label -> ground-truth test labels
y_te_map = {'A': y_test_A, 'B': y_test_B}

print('=== Scenario A: AUB-only ===')
oof_A, test_A, cv_f1_A = train_base_classifiers(X_train_A,  y_train_A,  X_test_A,  'A')

print('\n=== Scenario B: Merged (AUB+Peers) -> AUB test ===')
oof_B, test_B, cv_f1_B = train_base_classifiers(X_train_BC, y_train_BC, X_test_B,  'B')

## 5. Helper: Evaluate Ensemble

In [ ]:
results = []

def optimise_threshold(y_true, y_score, grid=np.arange(0.30, 0.71, 0.01)):
    best_thr, best_f1 = 0.5, 0.0
    for thr in grid:
        f1 = f1_score(y_true, (y_score >= thr).astype(int), zero_division=0)
        if f1 > best_f1:
            best_f1, best_thr = f1, thr
    return best_thr, best_f1

def evaluate(name, y_true, y_score, optimise_thr=True):
    thr = optimise_threshold(y_true, y_score)[0] if optimise_thr else 0.5
    y_pred = (y_score >= thr).astype(int)
    row = {
        'experiment': name,
        'f1':        round(f1_score(y_true, y_pred,          zero_division=0), 4),
        'auc':       round(roc_auc_score(y_true, y_score),                     4),
        'precision': round(precision_score(y_true, y_pred,   zero_division=0), 4),
        'recall':    round(recall_score(y_true, y_pred,      zero_division=0), 4),
        'threshold': round(thr, 2),
    }
    row['delta_f1']  = round(row['f1']  - BASELINE_F1,  4)
    row['delta_auc'] = round(row['auc'] - BASELINE_AUC, 4)
    results.append(row)
    sign = '+' if row['delta_f1'] >= 0 else ''
    print(f"  {name:<55} F1={row['f1']:.4f} ({sign}{row['delta_f1']:.4f})  "
          f"AUC={row['auc']:.4f}  thr={thr:.2f}")
    return row

print('Helpers defined.')

## 6. Experiments 1–2: Soft Voting (Averaged Probabilities)

Average the predicted probabilities from all base classifiers.
Simple, no training overhead, no risk of meta-learner overfit.

In [ ]:
print('=== Experiments 1-2: Soft Voting ===')

# Exp 1: Soft vote, Scenario A
soft_A = np.mean([test_A[m] for m in BASE_CLFS], axis=0)
evaluate('Exp1a: Soft vote all 4 classifiers (Scen A)', y_test_A, soft_A)

# Exp 2: Soft vote, Scenario B
soft_B = np.mean([test_B[m] for m in BASE_CLFS], axis=0)
evaluate('Exp1b: Soft vote all 4 classifiers (Scen B)', y_test_B, soft_B)

# Soft vote excluding RandomForest (often the weakest)
top3 = ['LogReg', 'XGBoost', 'LightGBM']
soft_A_top3 = np.mean([test_A[m] for m in top3], axis=0)
evaluate('Exp1c: Soft vote LogReg+XGB+LGBM (Scen A)',   y_test_A, soft_A_top3)

soft_B_top3 = np.mean([test_B[m] for m in top3], axis=0)
evaluate('Exp1d: Soft vote LogReg+XGB+LGBM (Scen B)',   y_test_B, soft_B_top3)

# Soft vote LogReg + XGBoost only (best two from individual results)
pair = ['LogReg', 'XGBoost']
evaluate('Exp1e: Soft vote LogReg+XGB only (Scen A)',    y_test_A,
         np.mean([test_A[m] for m in pair], axis=0))
evaluate('Exp1f: Soft vote LogReg+XGB only (Scen B)',    y_test_B,
         np.mean([test_B[m] for m in pair], axis=0))

## 7. Experiment 2: Weighted Soft Vote (Weight by CV F1)

In [ ]:
print('=== Experiments 2: Weighted Soft Voting ===')

def weighted_avg(probs_dict, weights_dict):
    total = sum(weights_dict.values())
    return sum(probs_dict[m] * (weights_dict[m] / total) for m in probs_dict)

# Scenario A — weight by OOF F1
wa_A = weighted_avg(test_A, cv_f1_A)
evaluate('Exp2a: Weighted soft vote by OOF F1 (Scen A)', y_test_A, wa_A)

wa_B = weighted_avg(test_B, cv_f1_B)
evaluate('Exp2b: Weighted soft vote by OOF F1 (Scen B)', y_test_B, wa_B)

# Scenario A — weight by OOF F1 squared (amplify stronger models)
cv_f1_A_sq = {k: v**2 for k, v in cv_f1_A.items()}
cv_f1_B_sq = {k: v**2 for k, v in cv_f1_B.items()}
evaluate('Exp2c: Weighted soft vote (F1-squared weights, Scen A)', y_test_A,
         weighted_avg(test_A, cv_f1_A_sq))
evaluate('Exp2d: Weighted soft vote (F1-squared weights, Scen B)', y_test_B,
         weighted_avg(test_B, cv_f1_B_sq))

print(f'\nOOF F1 weights (Scenario A): {cv_f1_A}')
print(f'OOF F1 weights (Scenario B): {cv_f1_B}')

## 8. Experiments 3–4: Stacking (OOF Probabilities -> LogReg Meta-Learner)

In [ ]:
print('=== Experiments 3-4: Stacking -> LogReg Meta-Learner ===')

def run_stacking(name, meta_train, y_tr, meta_test, y_te):
    meta_clf = Pipeline([
        ('scaler', StandardScaler()),
        ('lr',     LogisticRegression(max_iter=1000, random_state=RANDOM_STATE,
                                      class_weight='balanced'))
    ])
    meta_clf.fit(meta_train, y_tr)
    y_prob = meta_clf.predict_proba(meta_test)[:, 1]
    return evaluate(name, y_te, y_prob)

# All 4 classifiers -> meta-LogReg
run_stacking(
    'Exp3a: Stack all 4 classifiers -> LogReg (Scen A)',
    np.column_stack([oof_A[m]  for m in BASE_CLFS]), y_train_A,
    np.column_stack([test_A[m] for m in BASE_CLFS]), y_test_A
)
run_stacking(
    'Exp3b: Stack all 4 classifiers -> LogReg (Scen B)',
    np.column_stack([oof_B[m]  for m in BASE_CLFS]), y_train_BC,
    np.column_stack([test_B[m] for m in BASE_CLFS]), y_test_B
)

# Top 3 (LogReg + XGB + LGBM) -> meta-LogReg
run_stacking(
    'Exp3c: Stack LogReg+XGB+LGBM -> LogReg (Scen A)',
    np.column_stack([oof_A[m]  for m in top3]), y_train_A,
    np.column_stack([test_A[m] for m in top3]), y_test_A
)
run_stacking(
    'Exp3d: Stack LogReg+XGB+LGBM -> LogReg (Scen B)',
    np.column_stack([oof_B[m]  for m in top3]), y_train_BC,
    np.column_stack([test_B[m] for m in top3]), y_test_B
)

# Stacking + numeric features as extra meta-features
num_tr_A  = build_numeric_features(df_train_aub).fillna(0).values.astype('float32')
num_te_A  = build_numeric_features(df_test_aub).fillna(0).values.astype('float32')
num_tr_BC = build_numeric_features(df_train_all).fillna(0).values.astype('float32')
num_te_B  = build_numeric_features(df_test_aub_from_all).fillna(0).values.astype('float32')

run_stacking(
    'Exp3e: Stack all 4 + numeric -> LogReg (Scen A)',
    np.hstack([np.column_stack([oof_A[m]  for m in BASE_CLFS]), num_tr_A]),  y_train_A,
    np.hstack([np.column_stack([test_A[m] for m in BASE_CLFS]), num_te_A]),  y_test_A
)
run_stacking(
    'Exp3f: Stack all 4 + numeric -> LogReg (Scen B)',
    np.hstack([np.column_stack([oof_B[m]  for m in BASE_CLFS]), num_tr_BC]), y_train_BC,
    np.hstack([np.column_stack([test_B[m] for m in BASE_CLFS]), num_te_B]),  y_test_B
)

## 9. Experiment 5: Cross-Scenario Stacking (B classifiers -> A meta-LogReg)

Train base classifiers on merged data (larger, richer signal), but train
the meta-learner on AUB-only labels. Tests whether merged classifiers
produce better probability estimates even if their boundary is wrong.

In [ ]:
print('=== Experiment 5: Cross-scenario stacking (B classifiers -> A meta-LogReg) ===')

# We need B classifiers' predictions on the AUB-only training set.
# Align AUB feature space to the merged feature space first.
def align_columns(X_src, X_ref):
    out = X_src.copy()
    for c in set(X_ref.columns) - set(X_src.columns):
        out[c] = 0.0
    return out[X_ref.columns].astype('float32')

X_train_A_aligned = align_columns(X_train_A, X_train_BC)
X_test_A_aligned  = align_columns(X_test_A,  X_train_BC)
print(f'Aligned X_train_A: {X_train_A_aligned.shape} | X_test_A: {X_test_A_aligned.shape}')

oof_B_on_A  = {}
test_B_on_A = {}

for name, clf in BASE_CLFS.items():
    print(f'  Re-fitting {name} on merged data...', end=' ')
    clf_ = copy.deepcopy(clf)
    clf_.fit(X_train_BC, y_train_BC)
    oof_B_on_A[name]  = clf_.predict_proba(X_train_A_aligned)[:, 1]
    test_B_on_A[name] = clf_.predict_proba(X_test_A_aligned)[:, 1]
    print('done')

# Cross-scenario soft vote
evaluate('Exp5a: Cross-scenario soft vote (B->A)',
         y_test_A, np.mean([test_B_on_A[m] for m in BASE_CLFS], axis=0))

# Cross-scenario stacking
run_stacking(
    'Exp5b: Cross-scenario stack all 4 -> LogReg (B->A)',
    np.column_stack([oof_B_on_A[m]  for m in BASE_CLFS]), y_train_A,
    np.column_stack([test_B_on_A[m] for m in BASE_CLFS]), y_test_A
)

# Hybrid: mix Scenario A and Scenario B probabilities
# Average of same-model A and B predictions -> tests whether B adds diversity
hybrid_te = {m: (test_A[m] + test_B_on_A[m]) / 2 for m in BASE_CLFS}
hybrid_tr = {m: (oof_A[m]  + oof_B_on_A[m])  / 2 for m in BASE_CLFS}
evaluate('Exp5c: Hybrid soft vote (avg A+B per model)',
         y_test_A, np.mean(list(hybrid_te.values()), axis=0))
run_stacking(
    'Exp5d: Hybrid stack (avg A+B per model) -> LogReg',
    np.column_stack(list(hybrid_tr.values())), y_train_A,
    np.column_stack(list(hybrid_te.values())), y_test_A
)

## 10. Experiment 6: Domain-Aware Ensemble

The best individual result (nb42, F1=63.33%) used a domain-segmented approach:
a global model for most domains, domain-specific models only where they
meaningfully beat the global model (Social Sciences, Other).

Here: train a **full ensemble** within each domain and apply the same
selective override logic.

In [ ]:
print('=== Experiment 6: Domain-Aware Ensemble ===')

DOMAIN_MAP = {
    'Medicine':        ['Medicine', 'Nursing', 'Health', 'Pharmacol', 'Immunol', 'Dentistr'],
    'Engineering':     ['Engineer', 'Energy', 'Chemical', 'Material', 'Environ'],
    'Natural Sciences':['Physics', 'Chemistry', 'Earth', 'Biochem', 'Agri', 'Neuroscience'],
    'Computer Science':['Computer', 'Math', 'Decision'],
    'Social Sciences': ['Social', 'Psychol', 'Econom', 'Business', 'Arts', 'Education'],
    'Other':           [],
}

def assign_domain(df_):
    subj_col = next((c for c in df_.columns if 'subject' in c.lower() or 'asjc' in c.lower()), None)
    if subj_col is None:
        return pd.Series('Other', index=df_.index)
    subjects = df_[subj_col].fillna('').astype(str)
    domains  = pd.Series('Other', index=df_.index)
    for dom, keywords in DOMAIN_MAP.items():
        if not keywords: continue
        mask = subjects.str.contains('|'.join(keywords), case=False, na=False)
        domains[mask] = dom
    return domains

train_domains = assign_domain(df_train_aub)
test_domains  = assign_domain(df_test_aub)
print('Train domain distribution:')
print(train_domains.value_counts())

# Global soft vote (same as Exp1a but used as fallback)
global_probs_te = soft_A.copy()

# Domains where we'll override with a domain-specific ensemble
# (only if domain-specific soft vote > global soft vote on that domain's test data)
OVERRIDE_DOMAINS = ['Social Sciences', 'Other']  # from nb42 findings

final_probs = global_probs_te.copy()
test_index  = df_test_aub.index

for dom in OVERRIDE_DOMAINS:
    tr_mask = (train_domains == dom).values
    te_mask = (test_domains  == dom).values

    if tr_mask.sum() < 50:  # too few samples
        print(f'  {dom}: skip (only {tr_mask.sum()} train samples)')
        continue

    X_tr_dom = X_train_A.values[tr_mask]
    y_tr_dom = y_train_A[tr_mask]
    X_te_dom = X_test_A.values[te_mask]
    y_te_dom = y_test_A[te_mask]

    # Train ensemble on domain subset
    dom_probs_te = []
    for name, clf in BASE_CLFS.items():
        clf_ = copy.deepcopy(clf)
        clf_.fit(X_tr_dom, y_tr_dom)
        dom_probs_te.append(clf_.predict_proba(X_te_dom)[:, 1])
    dom_soft = np.mean(dom_probs_te, axis=0)

    # Compare with global probabilities on this domain's test slice
    global_f1_dom = f1_score(y_te_dom,
                             (global_probs_te[te_mask] >= 0.5).astype(int), zero_division=0)
    domain_f1_dom = f1_score(y_te_dom,
                             (dom_soft >= 0.5).astype(int), zero_division=0)
    print(f'  {dom}: global F1={global_f1_dom:.4f} | domain ensemble F1={domain_f1_dom:.4f}', end='')

    if domain_f1_dom > global_f1_dom:
        final_probs[te_mask] = dom_soft
        print(' --> OVERRIDE')
    else:
        print(' --> keep global')

evaluate('Exp6: Domain-aware ensemble (selective override)', y_test_A, final_probs)

## 11. Final Summary

In [ ]:
df_res = pd.DataFrame(results).set_index('experiment')
df_res = df_res[['f1', 'delta_f1', 'auc', 'delta_auc', 'precision', 'recall', 'threshold']]

baselines = pd.DataFrame([
    {'experiment': 'BASELINE: LogReg direct (threshold=0.54)',
     'f1': BASELINE_F1, 'delta_f1': 0.0,
     'auc': BASELINE_AUC, 'delta_auc': 0.0,
     'precision': 0.5258, 'recall': 0.7715, 'threshold': 0.54},
    {'experiment': 'BEST OVERALL: selective domain hybrid (nb42)',
     'f1': BEST_F1, 'delta_f1': round(BEST_F1 - BASELINE_F1, 4),
     'auc': None, 'delta_auc': None,
     'precision': None, 'recall': None, 'threshold': None},
]).set_index('experiment')

summary = pd.concat([baselines, df_res])

pd.set_option('display.max_colwidth', 60)
pd.set_option('display.float_format', lambda x: f'{x:+.4f}' if pd.notna(x) else 'N/A')

print('=' * 110)
print('CLASSIFICATION ENSEMBLE SUMMARY')
print('=' * 110)
print(summary.to_string())
print()

best_row = df_res.loc[df_res['f1'].idxmax()]
print('=' * 60)
print(f'BEST ENSEMBLE: {df_res["f1"].idxmax()}')
print(f"  F1  = {best_row['f1']:.4f}  (delta vs baseline: {best_row['delta_f1']:+.4f})")
print(f"  AUC = {best_row['auc']:.4f}  (delta vs baseline: {best_row['delta_auc']:+.4f})")
print('=' * 60)

n_beat = (df_res['f1'] > BASELINE_F1).sum()
n_beat_best = (df_res['f1'] > BEST_F1).sum()
print(f'\nExperiments beating LogReg baseline (F1>{BASELINE_F1}): {n_beat}/{len(df_res)}')
print(f'Experiments beating best overall (F1>{BEST_F1}):         {n_beat_best}/{len(df_res)}')

## 12. Visualisation

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

f1_vals  = df_res['f1'].values
auc_vals = df_res['auc'].values
labels   = df_res.index.tolist()

colors_f1  = ['#2ecc71' if v > BEST_F1 else '#f39c12' if v >= BASELINE_F1 else '#e74c3c'
               for v in f1_vals]
colors_auc = ['#2ecc71' if v >= BASELINE_AUC else '#e74c3c' for v in auc_vals]

for ax, vals, colors, ref1, ref1_label, ref2, ref2_label, xlabel, title in [
    (axes[0], f1_vals,  colors_f1,
     BASELINE_F1, f'LogReg baseline ({BASELINE_F1})',
     BEST_F1,     f'Best overall nb42 ({BEST_F1})',
     'F1 Score', 'Ensemble Experiments -- F1'),
    (axes[1], auc_vals, colors_auc,
     BASELINE_AUC, f'LogReg baseline ({BASELINE_AUC})',
     None, None,
     'ROC-AUC', 'Ensemble Experiments -- AUC'),
]:
    ax.barh(range(len(vals)), vals, color=colors, edgecolor='grey', linewidth=0.5)
    ax.axvline(ref1, color='steelblue', linestyle='--', lw=2, label=ref1_label)
    if ref2:
        ax.axvline(ref2, color='green', linestyle=':', lw=2, label=ref2_label)
    ax.set_yticks(range(len(labels)))
    ax.set_yticklabels(labels, fontsize=7)
    ax.set_xlabel(xlabel)
    ax.set_title(title)
    ax.legend(fontsize=8)
    for i, v in enumerate(vals):
        ax.text(v + 0.001, i, f'{v:.4f}', va='center', fontsize=7)

plt.tight_layout()
plt.savefig(REPORTS_DIR / 'nb64_ensemble_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved to reports/nb64_ensemble_results.png')

## 13. Individual Base Classifier Performance (Reference)

In [ ]:
print('Individual base classifier performance (threshold=0.5, no optimisation)')
print('=' * 70)
rows = []
for scen, test_probs, y_te in [('A', test_A, y_test_A), ('B', test_B, y_test_B)]:
    for name, probs in test_probs.items():
        thr, best_f1 = optimise_threshold(y_te, probs)
        rows.append({
            'model':    name,
            'scenario': scen,
            'f1_def':   round(f1_score(y_te, (probs >= 0.5).astype(int), zero_division=0), 4),
            'f1_opt':   round(best_f1, 4),
            'auc':      round(roc_auc_score(y_te, probs), 4),
            'opt_thr':  round(thr, 2),
        })
ind_df = pd.DataFrame(rows)
print(ind_df.to_string(index=False))

## 14. Conclusions

| Question | Answer |
|----------|--------|
| Does soft voting beat direct LogReg? | TBD |
| Does stacking classifiers help? | TBD |
| Does cross-scenario diversity add value? | TBD |
| Does domain-aware ensemble beat nb42's 63.33%? | TBD |

**Key structural difference from regression stacking (nb63)**:
Each base classifier here already optimises log-loss on the binary boundary.
Ensemble diversity comes from different inductive biases (linear vs tree vs forest)
rather than different objectives (MSE vs log-loss).

If no ensemble beats 63.33%, the project has reached the ceiling of this feature set
and the conclusion is clear: **better features, not better models, are the path forward**.